[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/32_topk_sampling.ipynb)

# 🟠 Medium: Top-k / Top-p (Nucleus) Sampling

Implement **sampling with top-k and top-p filtering** — the standard LLM decoding strategy.

### Signature
```python
def sample_top_k_top_p(logits, top_k=0, top_p=1.0, temperature=1.0) -> int:
    # logits: (V,) unnormalized log-probabilities
    # Returns: sampled token index
```

### Algorithm
1. Scale by temperature: `logits /= temperature`
2. Top-k: keep only top-k logits, set rest to `-inf`
3. Top-p: sort by prob, mask tokens where cumulative prob exceeds p
4. Sample from filtered distribution

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.4 MB/s eta 0:00:00


In [2]:
import torch

In [134]:
# ✏️ YOUR IMPLEMENTATION HERE

def sample_top_k_top_p(logits, top_k=0, top_p=1.0, temperature=1.0):
    pass  # temperature, top-k filter, top-p filter, sample
    logits = logits / max(temperature, 1e-8)

    if top_k > 0:
      top_k_logits, top_k_indices = torch.topk(logits, k=top_k, dim=-1)
      top_k_logit = top_k_logits[-1]
      logits = torch.where(logits < top_k_logit, float('-inf'), logits)

    sorted_logits, sorted_indices = logits.sort(descending=True)
    probs = torch.softmax(sorted_logits, dim=-1)

    if top_p < 1.0:
      cumsum = torch.cumsum(probs, dim=-1)
      mask = (cumsum - probs) > top_p
      sorted_logits[mask] = float('-inf')
      probs = torch.softmax(sorted_logits, dim=-1)

    sample = torch.multinomial(probs, 1)
    return sorted_indices[sample]

In [98]:
logits = torch.tensor([1.0, 5.0, 2.0, 0.5])
top_k = 0
top_p = 0.97

if top_k > 0:
  top_k_logits, top_k_indices = torch.topk(logits, k=top_k, dim=-1)
  top_k_logit = top_k_logits[-1]
  logits = torch.where(logits < top_k_logit, float('-inf'), logits)

In [99]:
logits

tensor([1.0000, 5.0000, 2.0000, 0.5000])

In [100]:
logits, indices = logits.sort(descending=True)
probs = torch.softmax(logits, dim=-1)

In [101]:
logits, probs

(tensor([5.0000, 2.0000, 1.0000, 0.5000]),
 tensor([0.9266, 0.0461, 0.0170, 0.0103]))

In [106]:
remove_mask = torch.cumsum(probs, dim=-1) > top_p
remove_mask

tensor([False,  True,  True,  True])

In [108]:
remove_mask[1:] = remove_mask[:-1].clone()
remove_mask[0] = False
remove_mask

tensor([False, False,  True,  True])

In [115]:
torch.where(remove_mask, float('-inf'), logits)

tensor([5., 2., -inf, -inf])

In [97]:
probs = torch.softmax(logits, dim=-1)
probs

tensor([0.9266, 0.0461, 0.0170, 0.0103])

In [30]:
mask = (torch.cumsum(probs, dim=-1) < top_p)
mask

tensor([ True, False, False, False])

In [31]:
mask[1:] = mask[:-1].clone()
mask[0] = True

In [32]:
mask

tensor([ True,  True, False, False])

In [36]:
top_p_logits_mask = torch.where(mask, 0, float('-inf'))
top_p_logits_mask

tensor([0., 0., -inf, -inf])

In [37]:
logits

tensor([5., 2., 1., -inf])

In [38]:
logits += top_p_logits_mask
logits

tensor([5., 2., -inf, -inf])

In [40]:
indices

tensor([1, 2, 0, 3])

In [216]:
relevant_indices = indices[:len(top_p_logits)]
relevant_indices

tensor([1, 2])

In [39]:
top_p_logits_probs = torch.softmax(logits, dim=-1)
top_p_logits_probs

tensor([0.9526, 0.0474, 0.0000, 0.0000])

In [218]:
sample = torch.multinomial(top_p_logits_probs, 1)
sample

tensor([0])

In [219]:
relevant_indices[sample]

tensor([1])

In [88]:
top_k_logits

tensor([5.])

In [63]:
logits = torch.tensor([1.0, 5.0, 2.0, 0.5])
logits = logits.sort(descending=True).values
probs = torch.softmax(logits, dim=-1)
probs

tensor([0.9266, 0.0461, 0.0170, 0.0103])

In [64]:
torch.cumsum(probs, dim=-1)

tensor([0.9266, 0.9727, 0.9897, 1.0000])

In [66]:
mask = torch.ones_like(logits)
mask = torch.where(torch.cumsum(probs, dim=-1) <= 0.98)
mask

(tensor([0, 1]),)

In [67]:
logits[mask]

tensor([5., 2.])

In [135]:
# 🧪 Debug
logits = torch.tensor([1.0, 5.0, 2.0, 0.5])
print('top_k=1:', sample_top_k_top_p(logits.clone(), top_k=1))
print('top_p=0.5:', sample_top_k_top_p(logits.clone(), top_p=0.5))
print('temp=0.01:', sample_top_k_top_p(logits.clone(), temperature=0.01))

top_k=1: tensor([1])
top_p=0.5: tensor([1])
temp=0.01: tensor([1])


In [136]:
# ✅ SUBMIT
from torch_judge import check
check('topk_sampling')


🧪 Testing: Top-k / Top-p Sampling (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] top_k=1 always returns argmax (7.3ms)
  ✅ [2/4] Low temperature concentrates (13.8ms)
  ❌ [3/4] All tokens reachable (no filtering)
     Only saw {tensor([0]), tensor([1]), tensor([0]), tensor([3]), tensor([1]), tensor([2]), tensor([3]), tensor([2]), tensor([1]), tensor([4]), tensor([4]), tensor([1]), tensor([2]), tensor([3]), tensor([2]), tensor([4]), tensor([2]), tensor([3]), tensor([3]), tensor([2]), tensor([1]), tensor([0]), tensor([0]), tensor([3]), tensor([3]), tensor([4]), tensor([2]), tensor([1]), tensor([0]), tensor([1]), tensor([2]), tensor([1]), tensor([0]), tensor([3]), tensor([1]), tensor([4]), tensor([4]), tensor([3]), tensor([4]), tensor([2]), tensor([1]), tensor([4]), tensor([2]), tensor([4]), tensor([4]), tensor([2]), tensor([0]), tensor([0]), tensor([0]), tensor([3]), tensor([3]), tensor([2]), tensor([2]), tensor([2]), tensor([0]), tensor([4]), tensor([0]), tensor(